In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import torch
import nibabel as nib
import os
import tensorflow as tf
from tensorflow.keras.utils import to_categorical # type: ignore # type: ignore
from skimage.transform import resize

In [2]:
print(pd.__version__)
print(np.__version__)
print(tf.__version__)
print(torch.__version__)
print("GPU is", "available" if torch.cuda.is_available() else "NOT AVAILABLE")

3.0.3
1.26.4
2.16.2
2.5.1+cpu
GPU is NOT AVAILABLE


In [4]:
from utils import util
IMG_SIZE = (128, 128)

VOLUME_DIR = util.get_volume_dir()
SEGMENTATION_DIR = util.get_segment_dir()

print(f"path:{os.listdir(SEGMENTATION_DIR)} {os.listdir(VOLUME_DIR)}")


X_train, Y_train = [], []

image_files = sorted([f for f in os.listdir(VOLUME_DIR) if f.endswith(".nii")])
mask_files = sorted([f for f in os.listdir(SEGMENTATION_DIR) if f.endswith(".nii")])

for img_file, mask_file in zip(image_files, mask_files):
    volume_path = os.path.join(VOLUME_DIR, img_file)
    segmentation_path = os.path.join(SEGMENTATION_DIR, mask_file)
    vol, seg = util.preprocess_data(volume_path, segmentation_path)
    X_train.append(vol)
    Y_train.append(seg)

X_train = np.concatenate(X_train, axis=0)
Y_train = np.concatenate(Y_train, axis=0)

print("Training data shape:", X_train.shape)
print("Segmentation mask shape:", Y_train.shape)

path:['segmentation-0.nii', 'segmentation-1.nii', 'segmentation-11.nii', 'segmentation-12.nii', 'segmentation-13.nii', 'segmentation-14.nii', 'segmentation-16.nii', 'segmentation-17.nii', 'segmentation-18.nii', 'segmentation-19.nii', 'segmentation-2.nii', 'segmentation-20.nii', 'segmentation-21.nii', 'segmentation-22.nii', 'segmentation-23.nii', 'segmentation-3.nii', 'segmentation-4.nii'] ['volume-0.nii', 'volume-1.nii', 'volume-11.nii', 'volume-12.nii', 'volume-13.nii', 'volume-14.nii', 'volume-16.nii', 'volume-17.nii', 'volume-18.nii', 'volume-19.nii', 'volume-2.nii', 'volume-20.nii', 'volume-21.nii', 'volume-22.nii', 'volume-23.nii', 'volume-3.nii', 'volume-4.nii']
Loading file: c:\Users\kreddy\Documents\Projects\LiverTumorSegmentation\ml_model\dataset\train\volumes\volume-0.nii
Loading file: c:\Users\kreddy\Documents\Projects\LiverTumorSegmentation\ml_model\dataset\train\segmentation\segmentation-0.nii
Loading file: c:\Users\kreddy\Documents\Projects\LiverTumorSegmentation\ml_model

In [7]:
import os
import numpy as np
import nibabel as nib
from tensorflow.keras.utils import to_categorical # type: ignore
from utils import util
# ---- Load Test Data ----
test_image_path = util.get_test_volume_dir()
test_mask_path = util.get_test_segment_dir()

# Ensure directories exist
if not os.path.exists(test_image_path) or not os.path.exists(test_mask_path):
    raise FileNotFoundError("Test data folder or segmentation folder not found!")

# Get sorted filenames
image_files = sorted([f for f in os.listdir(test_image_path) if f.endswith(".nii")])
mask_files = sorted([f for f in os.listdir(test_mask_path) if f.endswith(".nii")])

# Check if the number of images and masks match
if len(image_files) != len(mask_files):
    raise ValueError("Mismatch between the number of test images and segmentation masks!")

X_test, Y_test = [], []

# Load and preprocess each test image and mask
for img_file, mask_file in zip(image_files, mask_files):
    volume_path = os.path.join(test_image_path, img_file)  # Correct path joining
    segmentation_path = os.path.join(test_mask_path, mask_file)  # Correct path joining

    vol, seg = util.preprocess_data(volume_path, segmentation_path)
    X_test.append(vol)
    Y_test.append(seg)

# Convert lists to NumPy arrays
X_test = np.concatenate(X_test, axis=0)  # Shape: (num_volumes, num_slices, 128, 128)
Y_test = np.concatenate(Y_test, axis=0)  # Shape: (num_volumes, num_slices, 128, 128, 3)

# Reshape to flatten across all slices
X_test = X_test.reshape(-1, 128, 128, 1)  # Add channel dimension
Y_test = Y_test.reshape(-1, 128, 128, 3)  # Keep segmentation masks in one-hot format

# Print shapes
print("Final Test Data Shape:", X_test.shape)  # (total_slices, 128, 128, 1)
print("Final Test Mask Shape:", Y_test.shape)  # (total_slices, 128, 128, 3)


Loading file: c:\Users\kreddy\Documents\Projects\LiverTumorSegmentation\ml_model\dataset\test\volumes\volume-24.nii
Loading file: c:\Users\kreddy\Documents\Projects\LiverTumorSegmentation\ml_model\dataset\test\segmentation\segmentation-24.nii
Loading file: c:\Users\kreddy\Documents\Projects\LiverTumorSegmentation\ml_model\dataset\test\volumes\volume-25.nii
Loading file: c:\Users\kreddy\Documents\Projects\LiverTumorSegmentation\ml_model\dataset\test\segmentation\segmentation-25.nii
Final Test Data Shape: (877, 128, 128, 1)
Final Test Mask Shape: (877, 128, 128, 3)


In [ ]:
import torch
from torch.utils.data import DataLoader, Dataset

# Assuming x_train is (2090, 128, 128) and y_train is (2090, 128, 128, 3)
class SliceDataset(Dataset):
    def __init__(self, x, y):
        super().__init__()
        # Convert to float32 and torch tensors
        self.x = torch.tensor(x, dtype=torch.float32).unsqueeze(1)  # (N, 1, 128, 128)
        self.y = torch.tensor(y, dtype=torch.float32).permute(0, 3, 1, 2)  # (N, 3, 128, 128)

    def __len__(self):
        return self.x.shape[0]

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

# Create dataset and loader
batch_size = 16
dataset = SliceDataset(X_train, Y_train)
train_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

In [12]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from models.TransUNet import TransUNet

# -----------------------------
# Dataset
# -----------------------------
class SliceDataset(Dataset):
    def __init__(self, x, y):
        super().__init__()

        x = np.asarray(x)
        y = np.asarray(y)

        # Fix input shape: (N, H, W, 1) -> (N, H, W)
        if x.ndim == 4 and x.shape[-1] == 1:
            x = np.squeeze(x, axis=-1)
        elif x.ndim == 5 and x.shape[-1] == 1:
            x = np.squeeze(x, axis=-1)

        # Fix target shape: (N, H, W, 3) -> (N, 3, H, W)
        if y.ndim == 4 and y.shape[-1] == 3:
            y = np.transpose(y, (0, 3, 1, 2))

        self.x = torch.tensor(x, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

        # Ensure input has channel dim: (N, 1, H, W)
        if self.x.ndim == 3:
            self.x = self.x.unsqueeze(1)

        # Ensure target is in (N, C, H, W) format
        if self.y.ndim == 3:
            self.y = self.y.unsqueeze(1)

    def __len__(self):
        return self.x.shape[0]

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

# Create loader
batch_size = 16
dataset = SliceDataset(X_train, Y_train)
train_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# -----------------------------
# Losses
# -----------------------------
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, targets):
        # logits: (B, C, H, W)
        # targets: (B, H, W) or (B, 1, H, W)
        probs = F.softmax(logits, dim=1)

        if targets.dim() == 4:
            targets = targets.squeeze(1)

        targets_one_hot = F.one_hot(targets.long(), num_classes=logits.shape[1]) \
            .permute(0, 3, 1, 2).float()

        dims = (0, 2, 3)
        intersection = torch.sum(probs * targets_one_hot, dim=dims)
        union = torch.sum(probs + targets_one_hot, dim=dims)
        dice = (2.0 * intersection + self.smooth) / (union + self.smooth)
        return 1.0 - dice.mean()

# -----------------------------
# Training
# -----------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
model = TransUNet().to(device)

ce_loss = nn.CrossEntropyLoss()
dice_loss = DiceLoss(smooth=1.0)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

save_dir = os.path.join("checkpoints", "transUNet_experiment")
os.makedirs(save_dir, exist_ok=True)

num_epochs = 1
best_loss = float("inf")
best_epoch = -1

print(f"Training started on device: {device}")
print(f"Dataset size: {len(train_loader.dataset)}")
print(f"Number of batches: {len(train_loader)}")

for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0.0

    print(f"\n=== Epoch {epoch + 1}/{num_epochs} started ===")

    for batch_idx, (inputs, targets) in enumerate(train_loader):
        inputs = inputs.to(device).float()
        targets = targets.to(device).float()

        # Convert one-hot masks to class indices
        if targets.ndim == 4 and targets.shape[1] == 3:
            targets_idx = torch.argmax(targets, dim=1)
        else:
            targets_idx = targets.long()

        outputs = model(inputs)
        loss = ce_loss(outputs, targets_idx.long()) + dice_loss(outputs, targets_idx.long())

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

        # Print every 5 batches so you can see progress without too much output
        if batch_idx % 5 == 0 or batch_idx == len(train_loader) - 1:
            print(
                f"Epoch {epoch + 1}/{num_epochs} | "
                f"Batch {batch_idx + 1}/{len(train_loader)} | "
                f"Loss: {loss.item():.4f}"
            )

    avg_loss = epoch_loss / len(train_loader)
    print(f"Epoch {epoch + 1}/{num_epochs} finished | Average loss: {avg_loss:.4f}")

    # Save checkpoint once per epoch
    model_path = os.path.join(save_dir, f"epoch_{epoch + 1}.pth")
    torch.save(model.state_dict(), model_path)

    if avg_loss < best_loss:
        best_loss = avg_loss
        best_epoch = epoch + 1
        best_model_path = os.path.join(save_dir, "best_model.pth")
        torch.save(model.state_dict(), best_model_path)

print(f"\nTraining finished. Best epoch: {best_epoch} | Best loss: {best_loss:.4f}")

Training started on device: cpu
Dataset size: 8760
Number of batches: 548

=== Epoch 1/1 started ===
Epoch 1/1 | Batch 1/548 | Loss: 2.2896
Epoch 1/1 | Batch 6/548 | Loss: 1.8995
Epoch 1/1 | Batch 11/548 | Loss: 0.9347
Epoch 1/1 | Batch 16/548 | Loss: 0.7474
Epoch 1/1 | Batch 21/548 | Loss: 0.7119
Epoch 1/1 | Batch 26/548 | Loss: 0.7383
Epoch 1/1 | Batch 31/548 | Loss: 0.8481
Epoch 1/1 | Batch 36/548 | Loss: 0.8366
Epoch 1/1 | Batch 41/548 | Loss: 0.7025
Epoch 1/1 | Batch 46/548 | Loss: 0.7791
Epoch 1/1 | Batch 51/548 | Loss: 0.7994
Epoch 1/1 | Batch 56/548 | Loss: 0.7052
Epoch 1/1 | Batch 61/548 | Loss: 0.7724
Epoch 1/1 | Batch 66/548 | Loss: 0.7602
Epoch 1/1 | Batch 71/548 | Loss: 0.7579
Epoch 1/1 | Batch 76/548 | Loss: 0.7823
Epoch 1/1 | Batch 81/548 | Loss: 0.7950
Epoch 1/1 | Batch 86/548 | Loss: 0.7126
Epoch 1/1 | Batch 91/548 | Loss: 0.8967
Epoch 1/1 | Batch 96/548 | Loss: 0.8308
Epoch 1/1 | Batch 101/548 | Loss: 0.8323
Epoch 1/1 | Batch 106/548 | Loss: 0.6451
Epoch 1/1 | Batch 1

In [13]:
import os
import torch

save_dir = os.path.join(
    util.get_segment_model_dir(),
    "checkpoints/transUnet"
)
os.makedirs(save_dir, exist_ok=True)

save_path = os.path.join(save_dir, "transUnet_liver_segmentation.pth")

torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "num_epochs": num_epochs,
    "device": device
}, save_path)

print(f"Model saved to: {save_path}")

Model saved to: c:\Users\kreddy\Documents\Projects\LiverTumorSegmentation\ml_model\notebooks\models\checkpoints/transUnet\transUnet_liver_segmentation.pth


In [14]:
# Print shapes
X_test = np.squeeze(X_test, axis=-1)  # Now shape: (501, 128, 128)
print("Final Test Data Shape:", X_test.shape)  # (total_slices, 128, 128, 1)
print("Final Test Mask Shape:", Y_test.shape)  # (total_slices, 128, 128, 3)

Final Test Data Shape: (877, 128, 128)
Final Test Mask Shape: (877, 128, 128, 3)


In [15]:
class TestSliceDataset(Dataset):
    def __init__(self, x, y):
        super().__init__()
        # Remove the last dimension if it's singleton (1)
        if x.shape[-1] == 1:
            x = np.squeeze(x, axis=-1)  # Now (N, 128, 128)

        self.x = torch.tensor(x, dtype=torch.float32).unsqueeze(1)  # (N, 1, 128, 128)
        self.y = torch.tensor(y, dtype=torch.float32).permute(0, 3, 1, 2)  # (N, 3, 128, 128)

    def __len__(self):
        return self.x.shape[0]

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

# Use it
test_dataset = TestSliceDataset(X_test, Y_test)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)
for images, masks in test_loader:
    print("Fixed input shape:", images.shape)  # ✅ Should be (16, 1, 128, 128)
    print("Fixed mask shape:", masks.shape)    # ✅ Should be (16, 3, 128, 128)
    
    # Forward pass
    outputs = model(images)
    print("Model output shape:", outputs.shape)
    break


Fixed input shape: torch.Size([16, 1, 128, 128])
Fixed mask shape: torch.Size([16, 3, 128, 128])
Model output shape: torch.Size([16, 3, 128, 128])


In [ ]:
# # Recreate the model architecture first
# model = TransUnet(in_channels=1, out_channels=3)

# # Load the weights
# model.load_state_dict(torch.load('unetplusplus_epoch_15.pth'))

# # Put it in eval mode if you're using it for inference
# model.eval()

In [16]:
def dice_score(preds, targets, epsilon=1e-6):
    # Assumes preds are one-hot or softmax outputs (batch, C, H, W)
    preds = torch.argmax(preds, dim=1)  # (batch, H, W)
    targets = torch.argmax(targets, dim=1)  # (batch, H, W)

    dice_total = 0
    for cls in range(3):  # For each class
        pred_cls = (preds == cls).float()
        target_cls = (targets == cls).float()
        
        intersection = (pred_cls * target_cls).sum()
        union = pred_cls.sum() + target_cls.sum()
        
        dice = (2. * intersection + epsilon) / (union + epsilon)
        dice_total += dice

    return dice_total / 3  # Average over 3 classes

total_dice = 0
num_batches = 0

with torch.no_grad():
    for images, masks in test_loader:
        outputs = model(images)  # Forward pass
        batch_dice = dice_score(outputs, masks)
        total_dice += batch_dice.item()
        num_batches += 1

avg_dice = total_dice / num_batches
print(f"Average Dice Score on Test Set: {avg_dice:.4f}")

Average Dice Score on Test Set: 0.8265


In [12]:
def dice_score_per_class(preds, targets, epsilon=1e-6):
    # Convert from softmax/one-hot to label maps
    preds = torch.argmax(preds, dim=1)    # (batch, H, W)
    targets = torch.argmax(targets, dim=1)  # (batch, H, W)

    class_dice_scores = []

    for cls in range(3):  # 3 classes
        pred_cls = (preds == cls).float()
        target_cls = (targets == cls).float()

        intersection = (pred_cls * target_cls).sum()
        union = pred_cls.sum() + target_cls.sum()

        dice = (2. * intersection + epsilon) / (union + epsilon)
        class_dice_scores.append(dice.item())

    return class_dice_scores  # List: [dice_class_0, dice_class_1, dice_class_2]

In [13]:
total_dice = [0.0, 0.0, 0.0]
num_batches = 0

with torch.no_grad():
    for images, masks in test_loader:
        outputs = model(images)
        dice_scores = dice_score_per_class(outputs, masks)
        for i in range(3):
            total_dice[i] += dice_scores[i]
        num_batches += 1

avg_dice_per_class = [d / num_batches for d in total_dice]
for i, score in enumerate(avg_dice_per_class):
    print(f"Average Dice Score for Class {i}: {score:.4f}")

Average Dice Score for Class 0: 0.9888
Average Dice Score for Class 1: 0.6250
Average Dice Score for Class 2: 0.7188


In [14]:
def dice_score_foreground(preds, targets, epsilon=1e-6):
    preds = torch.argmax(preds, dim=1)  # (B, H, W)
    targets = torch.argmax(targets, dim=1)  # (B, H, W)

    dice_scores = []

    for cls in [1, 2]:  # Only foreground classes
        pred_cls = (preds == cls).float()
        target_cls = (targets == cls).float()
        
        intersection = (pred_cls * target_cls).sum()
        union = pred_cls.sum() + target_cls.sum()
        
        dice = (2. * intersection + epsilon) / (union + epsilon)
        dice_scores.append(dice)

    return sum(dice_scores) / len(dice_scores)

# Compute combined Dice for foreground (classes 1 and 2)
total_dice_fg = 0
num_batches = 0

with torch.no_grad():
    for images, masks in test_loader:
        outputs = model(images)
        batch_dice_fg = dice_score_foreground(outputs, masks)
        total_dice_fg += batch_dice_fg.item()
        num_batches += 1

avg_dice_fg = total_dice_fg / num_batches
print(f"Average Dice Score for Foreground (Classes 1 & 2): {avg_dice_fg:.4f}")

Average Dice Score for Foreground (Classes 1 & 2): 0.6719
